# MCP (Model Context Protocol) demo

This notebook launches `agents/tools/mcp_server.py` as a real MCP stdio subprocess
and drives it through `agents.tools.mcp_client.MCPToolClient` -- the same code path
the researcher agent could use to reach tools via MCP instead of calling the
underlying Python functions directly.

Prerequisites: the `infra/docker-compose.yml` Postgres + Qdrant containers running,
and the sample corpus ingested (`python -m rag.ingest data/sample_docs`).

In [ ]:
import os
import sys

sys.path.insert(0, "..")
os.environ.setdefault("LLM_PROVIDER", "fake")
os.environ.setdefault("EMBEDDINGS_PROVIDER", "fake")

from agents.tools.mcp_client import MCPToolClient

## 1. Discover the tools the server exposes

In [ ]:
async def list_tools():
    async with MCPToolClient() as client:
        return await client.list_tools()

await list_tools()

## 2. `search_cases` -- hybrid vector + BM25 search over the ingested corpus

In [ ]:
async def demo_search():
    async with MCPToolClient() as client:
        return await client.call_tool(
            "search_cases",
            {"query": "How long must a force majeure event last before termination?", "top_k": 3},
        )

results = await demo_search()
for r in results:
    print(f"[{r['source']}#{r['chunk_index']}] score={r['score']:.4f}\n  {r['text'][:160]}...\n")

## 3. `extract_citations` -- pull every indexed chunk for a given source document

In [ ]:
async def demo_citations():
    async with MCPToolClient() as client:
        return await client.call_tool(
            "extract_citations", {"doc_ids": ["termination_rights_summary.md"]}
        )

await demo_citations()

## 4. `query_sql` -- read-only SQL access (write statements are rejected)

In [ ]:
async def demo_sql():
    async with MCPToolClient() as client:
        rows = await client.call_tool(
            "query_sql", {"statement": "SELECT id, title, status FROM sessions LIMIT 5"}
        )
        try:
            await client.call_tool("query_sql", {"statement": "DELETE FROM sessions"})
        except RuntimeError as exc:
            print("Blocked as expected:", exc)
        return rows

await demo_sql()